# 图像生成模型完整教程

本教程深入介绍 VAE、扩散模型、Stable Diffusion、ControlNet 的原理和实现。

## 目录
1. 环境准备
2. VAE 变分自编码器
3. DDPM/DDIM 扩散模型
4. Stable Diffusion
5. ControlNet
6. 模型对比与分析

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')
torch.manual_seed(42)

## 2. VAE 变分自编码器

In [ ]:
from vae import create_vae_model, vae_loss

vae = create_vae_model('small')
vae = vae.to(device)
print(f'VAE 参数量: {sum(p.numel() for p in vae.parameters()):,}')

In [ ]:
# 前向传播
images = torch.randn(4, 3, 256, 256).to(device)
with torch.no_grad():
    recon, mu, logvar = vae(images)

print(f'输入: {images.shape}, 重建: {recon.shape}')
print(f'潜在空间: mu={mu.shape}, logvar={logvar.shape}')

In [ ]:
# 计算损失
loss, recon_loss, kl_loss = vae_loss(recon, images, mu, logvar, kl_weight=0.00025)
print(f'总损失: {loss.item():.4f}')
print(f'重建损失: {recon_loss.item():.4f}')
print(f'KL 损失: {kl_loss.item():.4f}')

In [ ]:
# 从潜在空间采样
with torch.no_grad():
    samples = vae.sample(num_samples=4, device=device)
print(f'生成样本: {samples.shape}')

# 可视化
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    img = samples[i].cpu().permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Sample {i+1}')
plt.suptitle('VAE 随机采样')
plt.tight_layout()
plt.show()

In [ ]:
# 潜在空间插值
with torch.no_grad():
    z1 = torch.randn(1, vae.config.latent_dim).to(device)
    z2 = torch.randn(1, vae.config.latent_dim).to(device)
    interpolated = vae.interpolate(z1, z2, steps=5)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, ax in enumerate(axes):
    img = interpolated[i].cpu().permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'α={i/4:.2f}')
plt.suptitle('VAE 潜在空间插值')
plt.tight_layout()
plt.show()

## 3. DDPM/DDIM 扩散模型

In [ ]:
from diffusion import create_diffusion_model, NoiseScheduler, DDIMSampler

ddpm = create_diffusion_model('small')
ddpm = ddpm.to(device)
print(f'DDPM 参数量: {sum(p.numel() for p in ddpm.parameters()):,}')

In [ ]:
# 训练步骤
images = torch.randn(4, 3, 64, 64).to(device)
loss = ddpm.training_step(images)
print(f'训练损失: {loss.item():.4f}')

In [ ]:
# 噪声调度可视化
scheduler = NoiseScheduler(num_timesteps=1000, beta_schedule='linear')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(scheduler.betas.numpy())
axes[0].set_xlabel('时间步 t')
axes[0].set_ylabel('β_t')
axes[0].set_title('噪声调度 (Beta)')

axes[1].plot(scheduler.alphas_cumprod.numpy())
axes[1].set_xlabel('时间步 t')
axes[1].set_ylabel('ᾱ_t')
axes[1].set_title('累积 Alpha')
plt.tight_layout()
plt.show()

In [ ]:
# 前向扩散过程可视化
x0 = torch.randn(1, 3, 64, 64).to(device)
timesteps = [0, 250, 500, 750, 999]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, t in enumerate(timesteps):
    t_tensor = torch.tensor([t]).to(device)
    noise = torch.randn_like(x0)
    xt = scheduler.q_sample(x0, t_tensor, noise)
    
    img = xt[0].cpu().permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(f't={t}')

plt.suptitle('前向扩散过程 (逐步加噪)')
plt.tight_layout()
plt.show()

## 4. Stable Diffusion

In [ ]:
from stable_diffusion import create_sd_model

sd = create_sd_model('tiny')
sd = sd.to(device)
print(f'Stable Diffusion 参数量: {sum(p.numel() for p in sd.parameters()):,}')

In [ ]:
# 文本编码
input_ids = torch.randint(0, 49408, (1, 77)).to(device)
with torch.no_grad():
    text_emb = sd.text_encoder(input_ids)
print(f'文本嵌入: {text_emb.shape}')

In [ ]:
# UNet 去噪
latent_size = sd.config.image_size // sd.config.latent_scale_factor
latents = torch.randn(1, sd.config.latent_channels, latent_size, latent_size).to(device)
timestep = torch.tensor([500]).to(device)

with torch.no_grad():
    noise_pred = sd.unet(latents, timestep, text_emb)
print(f'潜在表示: {latents.shape}, 预测噪声: {noise_pred.shape}')

In [ ]:
# CFG 演示
def cfg_predict(model, latents, timestep, cond_emb, uncond_emb, guidance_scale=7.5):
    noise_uncond = model.unet(latents, timestep, uncond_emb)
    noise_cond = model.unet(latents, timestep, cond_emb)
    return noise_uncond + guidance_scale * (noise_cond - noise_uncond)

uncond_emb = torch.zeros_like(text_emb)
with torch.no_grad():
    guided = cfg_predict(sd, latents, timestep, text_emb, uncond_emb)
print(f'CFG 引导后: {guided.shape}')

## 5. ControlNet

In [ ]:
from controlnet import create_controlnet, ZeroConv

controlnet = create_controlnet('canny', model_size='small')
controlnet = controlnet.to(device)
print(f'ControlNet 参数量: {sum(p.numel() for p in controlnet.parameters()):,}')

In [ ]:
# 零卷积验证
zero_conv = ZeroConv(64, 64)
x = torch.randn(1, 64, 32, 32)
y = zero_conv(x)
print(f'零卷积输入均值: {x.abs().mean():.4f}')
print(f'零卷积输出均值: {y.abs().mean():.6f} (应接近0)')

In [ ]:
# 条件编码
control_image = torch.randn(1, 3, 512, 512).to(device)
timestep = torch.tensor([500]).to(device)

with torch.no_grad():
    features = controlnet(control_image, timestep)

print(f'控制特征数量: {len(features)}')
for i, f in enumerate(features):
    print(f'  特征 {i}: {f.shape}')

## 6. 模型对比

In [ ]:
models = {'VAE': vae, 'DDPM': ddpm, 'SD': sd, 'ControlNet': controlnet}

print('模型参数量对比:')
print('-' * 40)
params = []
for name, m in models.items():
    p = sum(x.numel() for x in m.parameters())
    params.append(p)
    print(f'{name:15s}: {p:>12,} ({p/1e6:.1f}M)')

plt.figure(figsize=(10, 5))
plt.bar(models.keys(), [p/1e6 for p in params], color=['blue', 'orange', 'green', 'red'])
plt.ylabel('参数量 (M)')
plt.title('模型参数量对比')
plt.tight_layout()
plt.show()

## 总结

| 模型 | 特点 | 适用场景 |
|:-----|:-----|:---------|
| VAE | 快速，潜在空间可解释 | 图像压缩、特征学习 |
| DDPM | 高质量，采样慢 | 无条件图像生成 |
| SD | 文本条件，高效 | 文本到图像生成 |
| ControlNet | 精确控制 | 可控图像生成 |